## 03 - bronze -> silver (clean, dedupe, SCD2)

- takes the bronze rows loaded after the `silver` watermark
- flattens each resource into typed columns (`fhir_lakehouse/silver.py`, one function per resource)
- dedupes: the same resource version arriving twice keeps only the newest copy
- SCD type 2 on the resource id. A new version row is only opened when a tracked column changes (sha256 over the business columns). The old row gets `effective_to` and `is_current = false`. If a batch holds several versions of the same id they are chained in order.

Added columns: `record_hash, effective_from, effective_to, is_current, scd_version, silver_loaded_at, silver_updated_at, load_run_id`

In [ ]:
resource_type = "Patient"
run_id = ""
config_path = "/lakehouse/default/Files/config/pipeline_config.json"
code_path = "/lakehouse/default/Files/code"
lakehouse_root = ""    # blank: use the default lakehouse attached to the notebook

In [ ]:
import json
import sys

if code_path and code_path not in sys.path:
    sys.path.insert(0, code_path)

from pyspark.sql import functions as F

from fhir_lakehouse.config import apply_lakehouse_root, get_resource, load_config, table_name
from fhir_lakehouse.control import StepLog, new_run_id
from fhir_lakehouse.silver import MODELS, bronze_to_silver
from fhir_lakehouse.storage import Lakehouse, get_spark

cfg = apply_lakehouse_root(load_config(config_path), lakehouse_root)
resource = get_resource(cfg, resource_type)
run_id = run_id or new_run_id()

spark = get_spark()
spark.conf.set("spark.sql.session.timeZone", cfg["spark"]["timezone"])
lh = Lakehouse(spark, cfg)

In [ ]:
step = StepLog(lh, cfg, run_id, "bronze_to_silver", resource["name"])
try:
    stats = bronze_to_silver(lh, cfg, resource["name"], run_id)
    step.finish("SUCCEEDED", rows_read=stats["rows_read"], rows_inserted=stats["rows_inserted"],
                rows_updated=stats["rows_closed"], details=stats)
except Exception as e:
    step.finish("FAILED", error=repr(e))
    raise

In [ ]:
# ids that have more than one version so far
key = MODELS[resource["name"]].key
silver = lh.read(table_name(cfg, "silver", resource["name"]))
changed = silver.groupBy(key).count().where("count > 1").select(key)
print("rows:", silver.count(), "| current:", silver.where("is_current").count(), "| ids with history:", changed.count())
(silver.join(changed, key)
   .select(key, "scd_version", "version_id", "effective_from", "effective_to", "is_current")
   .orderBy(key, "scd_version")
   .show(10, truncate=False))

In [ ]:
result = json.dumps(stats, default=str)
print(result)
try:
    notebookutils.notebook.exit(result)  # shows up as the activity output in the pipeline
except NameError:
    pass  # not running in Fabric